# Level 3 - The loop of Henle: a countercurrent multiplier
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/kidney-sim/blob/main/notebooks/03_loop_of_henle_countercurrent.ipynb)

**Goal:** see how a hairpin tube with salt pumps creates a gradient from 300 to 1200 mOsm/L.

In [ ]:
# --- Setup: run this cell first --------------------------------------------
import os, sys, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/kidney-sim.git"   # change if you forked the project
if "google.colab" in sys.modules and not os.path.exists("kidneysim"):
    if not os.path.exists("kidney-sim"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("kidney-sim")
for path in [".", ".."]:
    if os.path.exists(os.path.join(path, "kidneysim")):
        sys.path.insert(0, os.path.abspath(path))
        break

import numpy as np
import matplotlib.pyplot as plt
import kidneysim as ks
from kidneysim import viz2d, viz3d
from kidneysim.ui import sliders
print("kidneysim", ks.__version__, "| interactive 3D (plotly):", viz3d.HAS_PLOTLY)

## 1. Box model

Split the loop into $n$ levels. $D_i$ = descending limb (and tissue), $A_i$ = ascending limb. Repeat:

1. **Single effect** (pumps):  $m_i = \tfrac{D_i + A_i}{2}$, $\;D_i = m_i + \tfrac{\delta}{2}$, $\;A_i = m_i - \tfrac{\delta}{2}$
2. **Washout** by blood: $D_i \leftarrow D_i - w\,(D_i - 300)$ (same for $A_i$)
3. **Flow**: shift everything one box around the hairpin; fresh fluid at 300 mOsm/L enters.

The pumps can only make a difference of $\delta \approx 200$ mOsm/L *at one level*, but flow **multiplies** it along the loop.

In [ ]:
res = ks.countercurrent.multiplier(n_levels=10, delta=200, n_cycles=150)
print(f"tip of the loop: {res['tip']:.0f} mOsm/L, fluid leaving: {res['outflow']:.0f} mOsm/L")
viz2d.plot_countercurrent(res);

### 3D: the gradient growing, cycle after cycle

In [ ]:
viz3d.show(viz3d.countercurrent_surface(res))

## 2. Why do desert animals have long loops?

In [ ]:
lengths, tips = ks.countercurrent.tip_vs_loop_length()
viz2d.plot_loop_length(lengths, tips);

In [ ]:
def experiment(n_levels=10, delta=200.0, washout=0.02):
    res = ks.countercurrent.multiplier(int(n_levels), delta, n_cycles=200, washout=washout)
    viz2d.plot_countercurrent(res)
    plt.show()

sliders(experiment, n_levels=(2, 30, 1, 10), delta=(0, 300, 10, 200), washout=(0, 0.1, 0.005, 0.02))

## 3. Connect to Level 2
The tip value is the `osm_max` used by the tubule model. A weak loop (e.g. furosemide) → the kidney cannot concentrate urine.

In [ ]:
for delta in [200, 100, 50]:
    tip = ks.countercurrent.multiplier(delta=delta)["tip"]
    r = ks.tubule.run_nephron(ADH=1.0, osm_max=tip)
    print(f"pump strength {delta:3d} -> tip {tip:5.0f} mOsm/L -> max urine osmolarity {r['urine_osm']:5.0f} mOsm/L")

## Questions
1. What happens if the pumps stop ($\delta = 0$)?
2. What does washout represent physically? What would happen if blood flow in the medulla increased a lot?
3. Kangaroo rats can make urine of ~5000 mOsm/L. What would they need in this model?